# Malaria Parasite Detection from Cell ImagesTraining script for the custom CNN used in the viewer.Reproduces `ml_models/malaria/trained_model.h5`.**Architecture:** 3 Conv+MaxPool blocks (32 → 64 → 64 filters) + Flatten + Dense(128) + Dropout + Dense(1, sigmoid)**Input:** 130 x 130 x 3 (RGB)**Output:** Binary sigmoid — 0 = Parasitized, 1 = Uninfected**Dataset:** NIH Malaria Cell Images — https://www.kaggle.com/datasets/iarunava/cell-images-for-detecting-malaria (27,558 images)**Reference code:** `Malaria-Detection-CNN-master/Notebook/malaria_detection_using_cnn.py`---

## 1. Setup

In [ ]:
import tensorflow as tffrom tensorflow.keras.models import Sequentialfrom tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropoutfrom tensorflow.keras.preprocessing.image import ImageDataGeneratorimport matplotlib.pyplot as pltprint(f'TensorFlow: {tf.__version__}')print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 2. Configuration

In [ ]:
TRAIN_DIR = 'cell_images/train'VAL_DIR = 'cell_images/test'  # per the original notebook conventionIMG_SIZE = (130, 130)INPUT_SHAPE = (130, 130, 3)BATCH_SIZE = 16EPOCHS = 20OUTPUT_MODEL = 'trained_model.h5'

## 3. Data Loading with Augmentation

In [ ]:
datagen = ImageDataGenerator(    rotation_range=20,    width_shift_range=0.10,    height_shift_range=0.10,    rescale=1.0/255,    shear_range=0.1,    zoom_range=0.1,    horizontal_flip=True,    fill_mode='nearest',)train_gen = datagen.flow_from_directory(    TRAIN_DIR,    target_size=IMG_SIZE,    batch_size=BATCH_SIZE,    color_mode='rgb',    class_mode='binary',)validation_gen = datagen.flow_from_directory(    VAL_DIR,    target_size=IMG_SIZE,    batch_size=BATCH_SIZE,    class_mode='binary',    shuffle=False,    color_mode='rgb',)print(f'Class indices: {train_gen.class_indices}')# Expected: {'parasitized': 0, 'uninfected': 1}

## 4. Build Model (Custom CNN)

In [ ]:
model = Sequential([    Conv2D(filters=32, kernel_size=(3, 3), input_shape=INPUT_SHAPE, activation='relu'),    MaxPooling2D(pool_size=(2, 2)),    Conv2D(filters=64, kernel_size=(3, 3), activation='relu'),    MaxPooling2D(pool_size=(2, 2)),    Conv2D(filters=64, kernel_size=(3, 3), activation='relu'),    MaxPooling2D(pool_size=(2, 2)),    Flatten(),    Dense(128, activation='relu'),    Dropout(0.5),    Dense(1, activation='sigmoid'),])model.compile(    loss='binary_crossentropy',    optimizer='adam',    metrics=['accuracy'],)model.summary()

## 5. Train

In [ ]:
history = model.fit(    train_gen,    epochs=EPOCHS,    validation_data=validation_gen,)

## 6. Save Model

In [ ]:
model.save(OUTPUT_MODEL)print(f'Saved: {OUTPUT_MODEL}')

## 7. Training Curves

In [ ]:
plt.figure(figsize=(12, 4))plt.subplot(1, 2, 1)plt.plot(history.history['loss'], label='Training Loss')plt.plot(history.history['val_loss'], label='Validation Loss')plt.xlabel('Epoch')plt.ylabel('Loss')plt.legend()plt.subplot(1, 2, 2)plt.plot(history.history['accuracy'], label='Training Accuracy')plt.plot(history.history['val_accuracy'], label='Validation Accuracy')plt.xlabel('Epoch')plt.ylabel('Accuracy')plt.legend()plt.tight_layout()plt.show()

## 8. Evaluate on Validation Set

In [ ]:
results = model.evaluate(validation_gen)print(f'Validation loss: {results[0]:.4f}')print(f'Validation accuracy: {results[1]:.4f}')

## 9. Deploy to ViewerCopy the trained model to the project:```<project-root>/ml_models/malaria/trained_model.h5```The Flask viewer loads it automatically on first prediction request.**Note on prediction:** The model outputs sigmoid ∈ [0, 1]. The viewer interprets ≥0.5 as *Uninfected* and <0.5 as *Parasitized*, matching the training `class_indices` (alphabetical order).